In [1]:
import pika

In [2]:
connection = pika.BlockingConnection(
    pika.ConnectionParameters(host="localhost"),
)

In [3]:
channel = connection.channel()
channel.basic_qos(prefetch_count=10)

In [6]:
channel.exchange_declare('RHEED', exchange_type='direct')

<METHOD(['channel_number=1', 'frame_type=1', 'method=<Exchange.DeclareOk>'])>

In [8]:
# try to declare special stream queue
# https://www.rabbitmq.com/docs/streams
# channel.queueDeclare(
#   "my-stream",
#   true,         // durable
#   false, false, // not exclusive, not auto-delete
#   Collections.singletonMap("x-queue-type", "stream")
# );

# Map<String, Object> arguments = new HashMap<>();
# arguments.put("x-queue-type", "stream");
# arguments.put("x-max-length-bytes", 20_000_000_000); // maximum stream size: 20 GB
# arguments.put("x-stream-max-segment-size-bytes", 100_000_000); // size of segment files: 100 MB
# channel.queueDeclare(
#   "my-stream",
#   true,         // durable
#   false, false, // not exclusive, not auto-delete
#   arguments
# );

result = channel.queue_declare(
    queue = "RHEED-stream",
    passive = False,
    durable = True,
    exclusive = False,
    auto_delete = False,
    arguments = {
        "x-queue-type": "stream",
        "x-max-length-bytes": int(1e8),
        "x-stream-max-segment-size-bytes": int(1e7),
    },
)

In [10]:
channel.queue_bind(
    exchange='RHEED', 
    queue=result.method.queue, 
    routing_key="image"
)

<METHOD(['channel_number=1', 'frame_type=1', 'method=<Queue.BindOk>'])>

In [ ]:
# channel.basicQos(100); // QoS must be specified
# channel.basicConsume(
#   "my-stream",
#   false, # auto_ack
#   Collections.singletonMap("x-stream-offset", "first"), // "first" offset specification
#   (consumerTag, message) -> {
#     // message processing
#     // ...
#    channel.basicAck(message.getEnvelope().getDeliveryTag(), false); // ack is required
#   }, # arguments
#   consumerTag -> { });

In [ ]:
def callback(ch, method, properties, body):
    print(f" [x] {method.routing_key}:{body[:10]}")
    ch.basic_ack(delivery_tag = method.delivery_tag, multiple=False)

In [ ]:

channel.basic_consume(
    queue=result.method.queue, 
    on_message_callback=callback, 
    auto_ack=False,
    arguments={"x-stream-offset": "first"}
)

channel.start_consuming()